# TMol How-To Guide

A single, growing reference notebook for working with TMol. Each section is self-contained and can be run independently once the Setup cell below has run.

## 1. Basics

Load a structure from the RCSB, inspect it, change it, save it, and build batches of poses.

### Setup



In Colab, choose a GPU runtime and Run all. For local execution, follow the [installation guide](../docs/installation.md). This section fetches structures live from the RCSB, so no local fixture files are required.

The first call into any of TMol's native routines in a fresh environment JIT-compiles kernels and can take several minutes; later calls reuse the compiled cache and are fast.

Wheel install, imports, seeding, and device selection are not themselves part of what this notebook teaches, so they live in [notebook_setup.py](notebook_setup.py) and are pulled in with a single call.

In [1]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlretrieve

    urlretrieve(
        "https://raw.githubusercontent.com/uw-ipd/tmol/"
        "master/notebooks/notebook_setup.py",
        "notebook_setup.py",
    )

from notebook_setup import *  # noqa: F401,F403

device, param_db = setup()


device=cpu


### Loading Structures

#### Download a CIF file from RCSB

We use [1UBQ](https://www.rcsb.org/structure/1UBQ) (ubiquitin) as the example structure for this section. `fetch_cif` downloads the raw mmCIF text; it is not written to disk here, just held in memory.

In [ ]:
PDB_ID = "1UBQ"
cif_text = fetch_cif(PDB_ID)
print(f"downloaded {PDB_ID}")


downloaded 103220 characters for 1UBQ


#### Load a PoseStack from a CIF

A PoseStack represents a batch of molecular systems (in this case, just a single structure - ubiquitin). PoseStacks are optimized for compactness for efficient processing on the GPU. `pose_stack_from_cif` accepts a path or an in-memory buffer.

In [ ]:
ubq_pose_stack = pose_stack_from_cif(io.StringIO(ubq_cif_text), device)
print(f"n_poses={ubq_pose_stack.n_poses}, max_n_blocks={ubq_pose_stack.max_n_blocks}")


n_poses=1, max_n_blocks=76


#### Load a PoseStack with ligands

To load a PoseStack with ligands, you must specify `prepare_ligands=True` while loading the structure. Without this flag, any ligands will be silently ommitted from the resulting PoseStack. More details about this in section 4 (TODO: add a forward link/reference to the ligand section when we re-combine the notebooks).

In [ ]:
# Fetch a new PDB with ligands present
LIGAND_PDB_ID = "3PTB"
ligand_cif_text = fetch_cif(LIGAND_PDB_ID)

# Load the PDB with prepare_ligands=True
ligand_pose_stack = pose_stack_from_cif(io.StringIO(ligand_cif_text), device, prepare_ligands=True)

#### Stacking PoseStacks

`tmol` is optimized to work best on batches of structures. 

You can batch multiple structures into a single PoseStack, even if they are different structures.

In [10]:
multi_pdb_stack = PoseStackBuilder.from_poses([pose_stack, ligand_pose_stack], device)
print(
    f"n_poses={multi_pdb_stack.n_poses}, "
    f"max_n_blocks={multi_pdb_stack.max_n_blocks} "
    f"(members had {pose_stack.max_n_blocks} and {ligand_pose_stack.max_n_blocks} blocks)"
)


n_poses=2, max_n_blocks=225 (members had 76 and 225 blocks)


#### Stacking multiple copies of a Pose

The same function, given `[pose_stack] * n`, builds a batch of `n` independent copies. This is the standard way to collect several independent stochastic outcomes from one starting structure, such as running FastRelax or the packer several times with different random seeds.

In [11]:
n_copies = 4
repeat_stack = PoseStackBuilder.from_poses([pose_stack] * n_copies, device)
print(f"n_poses={repeat_stack.n_poses}, max_n_blocks={repeat_stack.max_n_blocks}")


n_poses=4, max_n_blocks=76


### Query the PoseStack for info

#### Get all sequences

In [ ]:
sequence = tmol.get_sequence(pose_stack)
print(sequence)

76 blocks:
MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG


#### Get ligands (3-letter code + SMILES) TODO


TODO - claude couldn't find any simple functions for doing this, and it wants to go back to a biotite array to get the SMILES info

In [5]:
def pose_block_types(pose_stack, pose_index=0):
    """The real (non-padding) block types of one pose, in block order."""
    pbt = pose_stack.packed_block_types
    real = pose_stack.block_type_ind64[pose_index] >= 0
    type_inds = pose_stack.block_type_ind64[pose_index, real].tolist()
    return [pbt.active_block_types[i] for i in type_inds]


ligand_cif_text = fetch_cif("3PTB")
ligand_pose_stack, ligand_build_context = pose_stack_from_cif(
    io.StringIO(ligand_cif_text),
    device,
    param_db=param_db,
    prepare_ligands=True,
    return_context=True,
)
ligand_codes = sorted(
    {
        bt.name3
        for bt in pose_block_types(ligand_pose_stack)
        if not bt.properties.polymer.is_polymer
    }
)

built_array = biotite_from_pose_stack(ligand_pose_stack, ligand_build_context.canonical_ordering)
for code in ligand_codes:
    ligand_atoms = built_array[built_array.res_name == code]
    try:
        smiles = ligand_smiles_from_atom_array(ligand_atoms, res_name=code)
    except Exception:
        smiles = "(no bonds to derive SMILES from, e.g. a bare ion)"
    print(f"{code}: {ligand_atoms.array_length()} atoms, SMILES={smiles}")


*** Open Babel Warning  in WriteMolecule
  No 2D or 3D coordinates exist. Stereochemical information will be stored using an Open Babel extension. To generate 2D or 3D coordinates instead use --gen2D or --gen3D.


BEN: 18 atoms, SMILES=NC(=[NH2+])c1ccccc1
CA: 1 atoms, SMILES=(no bonds to derive SMILES from, e.g. a bare ion)


/home/jflat06/rosetta/tmol/tmol/io/_pose_stack_from_biotite.py:1421: UserWarning: Renumbering chain A of the output AtomArray: biotite's CIF writer does not support numbering that decreases within a chain
  warnings.warn(


#### Query phi/psi angles using pdb_info TODO


TODO - this is too complicated, and it also starts involving pdb_info, which we haven't introduced as a concept yet. It also seems like maybe we should have the ability to get phi/psi via both the original indexing and the tmol indexing?

In [6]:
def block_for_residue(pose_stack, pose_index, chain, residue_number):
    """Resolve a (chain, author residue number) label to a block index."""
    chain_labels = pose_stack.pdb_info.chain_labels[pose_index]
    residue_labels = pose_stack.pdb_info.residue_labels[pose_index]
    matches = np.flatnonzero((chain_labels == chain) & (residue_labels == residue_number))
    if len(matches) == 0:
        raise ValueError(f"no block found for chain {chain!r} residue {residue_number}")
    return int(matches[0])


query_block = block_for_residue(pose_stack, 0, "A", 43)
block_type = pose_stack.block_type(0, query_block)
print(f"chain A residue 43 -> block {query_block} ({block_type.name3})")
print("available torsions:", tmol.get_torsion_names(pose_stack, 0, query_block))

phi = tmol.get_named_torsions(pose_stack, 0, query_block, "phi")
psi = tmol.get_named_torsions(pose_stack, 0, query_block, "psi")
print(f"phi={phi:.1f} deg, psi={psi:.1f} deg")


chain A residue 43 -> block 42 (LEU)
available torsions: ['phi', 'psi', 'omega', 'chi1', 'chi2']
phi=-103.6 deg, psi=130.2 deg


#### Stacked pose querying

In [ ]:
# Some functions can work on the whole pose stack
tmol.get_sequence(multi_pdb_stack)

In [ ]:
# Others you must extract the individual poses
for pose_index in range(multi_pdb_stack.n_poses):
    single = multi_pdb_stack.split(pose_index)
    single_sequence = tmol.get_sequence(single)
    print(f"pose {pose_index}: {len(single_sequence)} blocks, starts {single_sequence[:20]}...")


pose 0: 76 blocks, starts MQIFVKTLTGKTITLEVEPS...
pose 1: 225 blocks, starts IVGGYTCGANTVPYQVSLNS...


### Setting basic info in the PoseStack

#### Set a residue's sidechain chi angles

Torsion values can be set directly by name using `tmol.set_named_torsions`. This returns a new `PoseStack` with the updated coordinates from the moves.


TODO: it seems like claude is really overcomplicating this.

In [8]:
def first_block_with_chi1(pose_stack, pose_index):
    for block in range(pose_stack.max_n_blocks):
        if int(pose_stack.block_type_ind64[pose_index, block]) == -1:
            continue
        if "chi1" in tmol.get_torsion_names(pose_stack, pose_index, block):
            return block
    raise ValueError("no block with a chi1 torsion was found")


chi_block = first_block_with_chi1(pose_stack, 0)
before_chi1 = tmol.get_named_torsions(pose_stack, 0, chi_block, "chi1")
rotated_pose_stack = tmol.set_named_torsions(
    pose_stack, 0, chi_block, "chi1", before_chi1 + 30.0
)
after_chi1 = tmol.get_named_torsions(rotated_pose_stack, 0, chi_block, "chi1")
print(
    f"block {chi_block} ({pose_stack.block_type(0, chi_block).name3}) chi1: "
    f"{before_chi1:.1f} -> {after_chi1:.1f} deg"
)


block 0 (MET) chi1: 80.4 -> 110.4 deg


#### Modify a residue's identity

TODO: very complicated - claude's notes below:

There is no direct "set this block's identity" call -- changing a residue's type is only reachable through the packer, which treats it as a (deterministic, since we seed it) search rather than an in-place edit. The minimum needed to mutate one residue:

1. A `ScoreFunction`, built from the shared parameter database.
2. A `PackerTask` restricted so that only the target block (and a one-residue shell around it, so the annealer has more than a single position to consider) is repackable, and the target is further restricted to the one requested identity.
3. Rotamer samplers registered on the task -- the Dunbrack backbone-dependent rotamer library for the new identity, plus samplers that keep nearby residues at their current rotamer.

`pack_rotamers` then returns a new `PoseStack`; the input is left unchanged.

In [7]:
target_block = query_block  # chain A residue 43
requested_identity = "ALA"

score_function = beta2016_score_function(device, param_db=param_db)

task = PackerTask(pose_stack, PackerPalette())
shell = torch.zeros_like(task.is_real_block)
shell[0, target_block] = True
shell[0, target_block + 1] = True  # give the annealer a second position to consider
task.restrict_to_repacking(~shell)

target_mask = torch.zeros_like(shell)
target_mask[0, target_block] = True
task.restrict_absent_name3s({requested_identity}, target_mask)
task.disable_packing_by_block_mask(~shell)

task.add_conformer_sampler(create_dunbrack_sampler_from_database(param_db, device))
task.add_conformer_sampler(FixedAAChiSampler())
task.add_conformer_sampler(IncludeCurrentSampler())

torch.manual_seed(SEED)
mutated_pose_stack = pack_rotamers(pose_stack, score_function, task)
print(
    f"block {target_block}: "
    f"{pose_stack.block_type(0, target_block).name3} -> "
    f"{mutated_pose_stack.block_type(0, target_block).name3}"
)


block 42: LEU -> ALA


### Writing outputs - TODO

TODO - this section is using the old pdb writer. There are functions for writing through biotite/atomworks, but these have the limitation of only being able to output structures with the same chemical identity. So they are not a general solution for a write-pdb/cif one liner. How do we want to resolve this?

`write_pose_stack_pdb` saves a `PoseStack` to a PDB file. `tmol.view` renders a `PoseStack` (or a Biotite structure, or raw PDB text) in an interactive py3Dmol viewer.

In [9]:
import tempfile
from pathlib import Path

output_path = Path(tempfile.gettempdir()) / f"{PDB_ID.lower()}_mutated.pdb"
write_pose_stack_pdb(mutated_pose_stack, str(output_path))
print(f"wrote {output_path}")

viewer = tmol.view(mutated_pose_stack)
viewer.show()


wrote /tmp/1ubq_mutated.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.